In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,recall_score, precision_score, f1_score, roc_auc_score, classification_report, ConfusionMatrixDisplay
from src.data.load_data import import_dataset
from src.config import features, features_cat, features_num, target
from src.preprocess import churn_mapped, create_preprocessor, train_test, total_charges


Considerando as metricas tiradas dos modelos no nb 02, o modelo a ser escolhido é o de Regressão Logistica, pois possui a maior curva ROC-AUC (0.84) e precision/recall proximo de todos os modelos, com valores satisfatorios (0.78 e 0.5). Dessa forma, as analises levarão em consideração esses parametros. Mesmo com valores de modelos mais robustos como XGBoost, a RL mostra um desempenho semelhante em AUC-ROC e maior em Recall, sendo também um modelo mais simples e de fácil entendimento

In [4]:
df = import_dataset(r"..\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = total_charges(df)

In [5]:
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [42]:
df_churn = df.loc[df["Churn"]=='Yes']
lost = df_churn["MonthlyCharges"].sum()
qtd_churn = df_churn["Churn"].count()

In [13]:
lost

np.float64(139130.85)

In [57]:
print(f"Quantidade de Churn identificados: {qtd_churn*0.78}")
print(f"Quantidade possivel do impacto financeiro mensal : {lost*0.78}")
print(f"Quantidade possivel do impacto financeiro anual: {lost*0.78*12}")

Quantidade de Churn identificados: 1457.82
Quantidade possivel do impacto financeiro mensal : 108522.06300000001
Quantidade possivel do impacto financeiro anual: 1302264.756


In [58]:
print(f"Quantidade de Churn pós campanha: {qtd_churn*0.78*0.3}")
print(f"Quantidade possivel do impacto financeiro mensal : {lost*0.78*0.3}")
print(f"Quantidade possivel do impacto financeiro anual: {lost*0.78*12*0.3}")

Quantidade de Churn pós campanha: 437.34599999999995
Quantidade possivel do impacto financeiro mensal : 32556.6189
Quantidade possivel do impacto financeiro anual: 390679.4268


Se aplicarmos o modelo de regressão logistica, podemos prever 78% dos clientes que realmente deram Churn. Sendo assim, manteriamos 78% da receita mensal (~U$110) que, ao anualizarmos, teriamos  cerca de U$1300k. Se fizessemos uma campanha que recuperassemos 30% desses clientes, pensando em um cenario conservador, teriamos U$32k mensal e U$390K anual de retorno.

Para simulações, vamos considerar campanhas de 10, 15 e 20 dolares por clientes para entrar em contato e tentar a retenção:

In [36]:
find_churn = int(qtd_churn*0.50)
print(f"Quantidade de Churn encontrado: {find_churn}")

campanha = [10, 15, 20]
custo_campanha = []
for c in campanha:
    result_camp = int(find_churn*c)
    custo_campanha.append(result_camp)
print(custo_campanha)



Quantidade de Churn encontrado: 934
[9340, 14010, 18680]


Supondo que, dos 50% dos clientes que previmos que dessem Churn, conseguissemos reter 50%, teriamos 467 clientes, que traria um retorno médio de  35k. Considerando que a campanha mais cara teria um custo de 18,7k, então teriamos um retorno de 50% do valor no primeiro mês da campanha

In [56]:
churn_acetado = round(float(find_churn*0.5/qtd_churn),3)
churn_acetado *lost

np.float64(34782.7125)